In [726]:
pip install line_profiler

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [282]:
import librosa
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import laion_clap
import torch

model = laion_clap.CLAP_Module(enable_fusion=False)
model.load_ckpt()
model.eval()

torch.manual_seed(0)
np.random.seed(0)



Load our best checkpoint in the paper.
The checkpoint is already downloaded
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.we

CLAP_Module(
  (model): CLAP(
    (audio_branch): HTSAT_Swin_Transformer(
      (spectrogram_extractor): Spectrogram(
        (stft): STFT(
          (conv_real): Conv1d(1, 513, kernel_size=(1024,), stride=(480,), bias=False)
          (conv_imag): Conv1d(1, 513, kernel_size=(1024,), stride=(480,), bias=False)
        )
      )
      (logmel_extractor): LogmelFilterBank()
      (spec_augmenter): SpecAugmentation(
        (time_dropper): DropStripes()
        (freq_dropper): DropStripes()
      )
      (bn0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (patch_embed): PatchEmbed(
        (proj): Conv2d(1, 96, kernel_size=(4, 4), stride=(4, 4))
        (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (layers): ModuleList(
        (0): BasicLayer(
          dim=96, input_resolution=(64, 64), depth=2
          (blocks): ModuleList(
            (0): SwinTransformerBlock(
      

In [750]:
def get_clap_embedding_long(y: np.ndarray, sr: int, 
                             chunk_sec: int = 30) -> np.ndarray:
    """
    Делит длинное аудио на чанки по chunk_sec секунд,
    получает эмбеддинг каждого, усредняет.
    """
    chunk_len = sr * chunk_sec
    
    # Нарезаем на чанки
    chunks = []
    for start in range(0, len(y), chunk_len):
        chunk = y[start : start + chunk_len]
        
        # Пропускаем слишком короткие хвосты (< 5 секунд)
        if len(chunk) < sr * 5:
            continue
        
        # Padding до фиксированной длины
        if len(chunk) < chunk_len:
            chunk = np.pad(chunk, (0, chunk_len - len(chunk)))
        
        chunks.append(chunk)
    
    if not chunks:
        raise ValueError("Аудио слишком короткое")
    
    # Эмбеддинг каждого чанка
    batch = np.stack(chunks)  # shape: (n_chunks, chunk_len)
    
    with torch.no_grad():
        embeddings = model.get_audio_embedding_from_data(
            x=batch, use_tensor=False
        )  # shape: (n_chunks, 512)
    
    # Усредняем по чанкам
    return embeddings.mean(axis=0)  # shape: (512,)

def unit_norm(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v
    
# @profile
def extract_all_features(path: str, duration: int = 300, clap_weight: float = 0.5) -> np.ndarray:
    y, sr = librosa.load(path, duration=duration, sr=48000, mono=True)
    parts = []
    
    # Фиксируем длину
    target_len = int(sr * duration)
    y = y[:target_len] if len(y) >= target_len else np.pad(y, (0, target_len - len(y)))

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)[1:]
    parts.append(np.hstack([mfcc.mean(axis=1), mfcc.std(axis=1)]))
    
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    parts.append(np.hstack([chroma.mean(axis=1), chroma.std(axis=1)]))

    tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
    beat_times = librosa.frames_to_time(beat_frames, sr=sr)
    diffs = np.diff(beat_times)
    std = diffs.std()
    parts.append(np.array([std]))
    parts.append(np.array([float(tempo)]))

    bounds = librosa.segment.agglomerative(mfcc, k=5)
    bound_times = librosa.frames_to_time(bounds, sr=sr)
    structure_vec = np.array([
        len(bounds),                          # кол-во секций
        np.diff(bound_times).mean(),          # средняя длина секции
        np.diff(bound_times).std(),           # вариативность структуры
    ])
    parts.append(structure_vec)
    

    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    parts.append(np.array([centroid.mean(), centroid.std()]))

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    parts.append(np.array([rolloff.mean(), rolloff.std()]))


    # CLAP — чанками
    clap_vec = get_clap_embedding_long(y, sr, chunk_sec=30)

    # 7. Tempogram — локальные ритмические паттерны
    oenv = librosa.onset.onset_strength(y=y, sr=sr)
    tempogram = librosa.feature.tempogram(onset_envelope=oenv, sr=sr)
    rhythm_vec = tempogram.mean(axis=1)
    parts.append(np.array([tempogram.mean(), tempogram.std()]))
    parts.append(rhythm_vec)

    acousic = np.hstack([unit_norm(p) for p in parts])

    combined = np.hstack([
        acoustic * (1 - clap_weight),
        unit_norm(clap_vec) * clap_weight,
    ])
    return combined

def song_similarity(path_a: str, path_b: str, duration: int = 30, clap_weight: float = 0.5) -> float:
    fa = extract_all_features(path_a, duration, clap_weight).reshape(1, -1)
    fb = extract_all_features(path_b, duration, clap_weight).reshape(1, -1)
    return float(cosine_similarity(fa, fb)[0][0])  # 1.0 = identical, 0.0 = unrelated

In [775]:
path1 = r"S:\Music\Daft Punk - 2001 - Discovery [WPCR-80083]\04 - Harder, Better, Faster, Stronger.flac"
path2 = r"S:\Music\Kanye West ALAC\2007-09-05 - Graduation [JP - UICY-60010 - 2009]\03 Stronger.m4a"
path3 = r"S:\Music\Kanye West ALAC\2007-09-05 - Graduation [JP - UICY-60010 - 2009]\04 I Wonder.m4a"

In [753]:

score_stronger = song_similarity(path1, path2, duration=60, clap_weight=0.65)
score_wonder = song_similarity(path1, path3, duration=60, clap_weight=0.65)
print(f"Similarity (SHOULD BE HIGHER): {score_stronger:.3f}, (SHOULD BE LOWER): {score_wonder:.3f}")

Similarity (SHOULD BE HIGHER): 0.570, (SHOULD BE LOWER): 0.414


In [810]:
text='sad song with slow tempo'

In [764]:
path1 = r"S:\Music\Eminem - The Eminem Show (Expanded Edition) - 2002 [2022) [FLAC]\17. Say What You Say.flac"

In [773]:
def compare_with_query(query, path):
    with torch.no_grad():
        text_embedding = model.get_text_embedding(query)[0]  # shape: (3, 512)
        
    y, sr = librosa.load(path, duration=120, sr=48000, mono=True)
    music_vec = get_clap_embedding_long(y, sr, chunk_sec=30)
    return float(cosine_similarity(text_embedding.reshape(1, -1), music_vec.reshape(1, -1))[0][0])

In [828]:
compare_with_query(text, r"S:\Music\Lana Del Rey - Born To Die The Paradise Edition [Japan]\CD-1 Born To Die\Lana Del Rey - Born To Die The Paradise Edition_Track02.flac")

0.15520690381526947

In [494]:
y, sr = librosa.load(path2, duration=120, sr=48000, mono=True)

In [603]:
tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
beat_times = librosa.frames_to_time(beat_frames, sr=sr)

In [604]:
beat_times.std()

34.4893601103227

In [497]:
rhythm_vec

array([1.00000000e+00, 7.45657093e-01, 4.99872960e-01, 4.17940164e-01,
       3.90038350e-01, 3.74539088e-01, 3.65356292e-01, 3.53617392e-01,
       3.42973178e-01, 3.46835795e-01, 3.66133715e-01, 3.83028965e-01,
       3.88425642e-01, 3.88584306e-01, 3.94209589e-01, 4.11261673e-01,
       4.45087360e-01, 4.71025774e-01, 4.50337130e-01, 4.21113324e-01,
       3.99751957e-01, 3.82616044e-01, 3.69402872e-01, 3.62707482e-01,
       3.60783039e-01, 3.63429594e-01, 3.66414870e-01, 3.69974315e-01,
       3.83126714e-01, 4.04024206e-01, 4.08905225e-01, 3.99225004e-01,
       3.95319123e-01, 3.99735289e-01, 4.08574524e-01, 4.24607732e-01,
       4.41021301e-01, 4.14683474e-01, 3.74779385e-01, 3.58245796e-01,
       3.52144378e-01, 3.49744266e-01, 3.54180782e-01, 3.61349267e-01,
       3.67207939e-01, 3.93682420e-01, 4.41695050e-01, 4.27804396e-01,
       3.85308038e-01, 3.68018050e-01, 3.67541110e-01, 3.80508681e-01,
       4.08295729e-01, 4.33475610e-01, 3.94947329e-01, 3.54644056e-01,
      

In [488]:
onset_env = librosa.onset.onset_strength(y=y, sr=sr)

In [495]:
tempogram = librosa.feature.tempogram(onset_envelope=onset_env, sr=sr)
rhythm_vec = tempogram.mean(axis=1)  # усредняем по времени

In [496]:
len(rhythm_vec)

384